## جاسازی‌ها

در مثال قبلی ما، بر روی بردارهای کیسه کلمات با ابعاد بالا با طول `vocab_size` کار کردیم و به طور صریح از بردارهای موقعیتی با ابعاد پایین به نمایش یک-داغ پراکنده تبدیل می‌شدیم. این نمایش یک-داغ از نظر حافظه کارآمد نیست، علاوه بر این، هر کلمه به‌طور مستقل از سایر کلمات در نظر گرفته می‌شود، یعنی بردارهای کدگذاری‌شده یک-داغ هیچ شباهت معنایی بین کلمات را بیان نمی‌کنند.

در این بخش، به بررسی ادامه‌ای از مجموعه داده **News AG** می‌پردازیم. برای شروع، داده‌ها را بارگذاری کرده و برخی تعاریف از دفترچه قبلی را دریافت می‌کنیم.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## امبدینگ چیست؟

ایده **امبدینگ** این است که کلمات را با بردارهای متراکم با ابعاد پایین‌تر نمایش دهیم، که به نحوی معنای لغوی یک کلمه را منعکس می‌کند. بعداً درباره نحوه ساخت امبدینگ‌های معنادار کلمات صحبت خواهیم کرد، اما فعلاً اجازه دهید امبدینگ‌ها را روشی برای کاهش ابعاد بردار کلمات در نظر بگیریم.

بنابراین، لایه امبدینگ کلمه‌ای را به عنوان ورودی می‌گیرد و بردار خروجی‌ای با ابعاد مشخص شده `embedding_size` تولید می‌کند. به نوعی بسیار شبیه لایه `Linear` است، اما به جای گرفتن بردار وان-هات کدگذاری شده، می‌تواند شماره یک کلمه را به عنوان ورودی بپذیرد.

با استفاده از لایه امبدینگ به عنوان اولین لایه در شبکه خود، می‌توانیم از مدل کیسه کلمات به مدل **embedding bag** تغییر دهیم، جایی که ابتدا هر کلمه در متن را به امبدینگ متناظر تبدیل می‌کنیم، سپس تابع تجمیع خاصی را روی همه این امبدینگ‌ها محاسبه می‌کنیم، مانند `sum`، `average` یا `max`.

![تصویری که یک طبقه‌بند امبدینگ را برای پنج کلمه توالی نشان می‌دهد.](../../../../../translated_images/fa/embedding-classifier-example.b77f021a7ee67eee.webp)

شبکه عصبی طبقه‌بندی ما با لایه امبدینگ شروع می‌شود، سپس لایه تجمیع، و در نهایت طبقه‌بند خطی روی آن قرار دارد:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### برخورد با اندازه متغیر دنباله

به دلیل این معماری، مینی‌دسته‌ها برای شبکه ما باید به روشی خاص ساخته شوند. در واحد قبلی، هنگامی که از کیسه کلمات استفاده می‌کردیم، تمام تنسورهای BoW در یک مینی‌دسته اندازه برابر `vocab_size` داشتند، صرف‌نظر از طول واقعی دنباله متنی ما. هنگامی که به تعبیه کلمات منتقل می‌شویم، در هر نمونه متنی با تعداد متغیر کلمه روبه‌رو خواهیم شد و هنگام ترکیب آن نمونه‌ها در مینی‌دسته‌ها باید مقداری پدینگ اعمال کنیم.

این کار می‌تواند با استفاده از همان تکنیک ارائه تابع `collate_fn` به منبع داده انجام شود:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### آموزش طبقه‌بند تعبیه‌شده

اکنون که دیتالودر مناسب را تعریف کرده‌ایم، می‌توانیم مدل را با استفاده از تابع آموزش که در واحد قبلی تعریف کردیم، آموزش دهیم:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **توجه**: ما اینجا فقط برای ۲۵ هزار رکورد آموزش می‌دهیم (کمتر از یک دوره کامل) به خاطر صرفه‌جویی در زمان، اما می‌توانید آموزش را ادامه دهید، تابعی برای آموزش در چند دوره بنویسید، و با پارامتر نرخ یادگیری آزمایش کنید تا به دقت بالاتری برسید. باید بتوانید دقت را تا حدود ۹۰٪ افزایش دهید.


### لایه EmbeddingBag و نمایش دنباله‌های با طول متغیر

در معماری قبلی، لازم بود تا تمام دنباله‌ها را به یک طول مشخص پد کنیم تا بتوانیم آن‌ها را در یک مینی‌بچ جای دهیم. این روش بهینه‌ترین راه برای نمایش دنباله‌های با طول متغیر نیست - رویکرد دیگری استفاده از بردار **offset** است که آفست‌های همه دنباله‌ها را که در یک بردار بزرگ ذخیره شده‌اند، نگه می‌دارد.

![تصویری که نمایش دنباله با آفست را نشان می‌دهد](../../../../../translated_images/fa/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **توجه**: در تصویر بالا، دنباله‌ای از کاراکترها نشان داده شده است، اما در مثال ما با دنباله‌ای از کلمات کار می‌کنیم. با این حال، اصل کلی نمایش دنباله‌ها با بردار offset یکسان باقی می‌ماند.

برای کار با نمایش offset، ما از لایه [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) استفاده می‌کنیم. این لایه شبیه `Embedding` است، اما بردار محتوا و بردار offset را به عنوان ورودی می‌گیرد و همچنین شامل یک لایه میانگین‌گیری می‌شود که می‌تواند `mean`، `sum` یا `max` باشد.

اینجا شبکه تغییر یافته‌ای است که از `EmbeddingBag` استفاده می‌کند:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

برای آماده‌سازی مجموعه داده برای آموزش، باید یک تابع تبدیل ارائه دهیم که بردار آفست را آماده کند:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

توجه داشته باشید که برخلاف تمام مثال‌های قبلی، شبکه ما اکنون دو پارامتر می‌پذیرد: بردار داده و بردار آفست، که اندازه‌های متفاوتی دارند. به همین ترتیب، بارگذار داده ما نیز ۳ مقدار به جای ۲ مقدار به ما می‌دهد: هر دو بردار متن و آفست به عنوان ویژگی ارائه می‌شوند. بنابراین، ما نیاز داریم تابع آموزش‌مان را کمی تنظیم کنیم تا این موضوع را در نظر بگیرد:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## تعبیه‌های معنایی: Word2Vec

در مثال قبلی ما، لایه تعبیه مدل یاد گرفت که کلمات را به نمایش برداری نگاشت کند، اما این نمایش معنای چندانی نداشت. خوب است که چنین نمایش برداری یاد گرفته شود که کلمات مشابه یا مترادف‌ها به بردارهایی متناظر باشند که از نظر فاصله برداری (مثلاً فاصله اقلیدسی) به یکدیگر نزدیک باشند.

برای این کار، باید مدل تعبیه خود را به روشی خاص روی مجموعه بزرگی از متن‌ها پیش‌آموزش دهیم. یکی از اولین روش‌ها برای آموزش تعبیه‌های معنایی، [Word2Vec](https://en.wikipedia.org/wiki/Word2vec) نام دارد. این روش بر دو معماری اصلی مبتنی است که برای تولید نمایش توزیع‌شده کلمات به کار می‌رود:

 - **کیسه کلمات پیوسته** (CBoW) — در این معماری، مدل آموزش داده می‌شود که یک کلمه را از زمینه‌های اطراف پیش‌بینی کند. با داشتن ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$، هدف مدل پیش‌بینی $W_0$ از $(W_{-2},W_{-1},W_1,W_2)$ است.
 - **پردازش پیوسته skip-gram** برعکس CBoW است. مدل از پنجره‌ای از کلمات زمینه اطراف برای پیش‌بینی کلمه جاری استفاده می‌کند.

CBoW سریع‌تر است، در حالی که skip-gram کندتر است، اما کار بهتری در نمایش کلمات کم‌تکرار انجام می‌دهد.

![تصویری که هر دو الگوریتم CBoW و Skip-Gram را برای تبدیل کلمات به بردارها نشان می‌دهد.](../../../../../translated_images/fa/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

برای آزمایش تعبیه word2vec پیش‌آموزش شده روی داده‌های Google News می‌توانیم از کتابخانه **gensim** استفاده کنیم. در زیر کلماتی که بیشترین شباهت را به 'neural' دارند می‌یابیم

> **توجه:** وقتی اولین بار بردارهای کلمات را ایجاد می‌کنید، دانلود آن‌ها ممکن است مدتی طول بکشد!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


ما همچنین می‌توانیم جاسازی‌های برداری را از کلمه محاسبه کنیم تا در آموزش مدل طبقه‌بندی استفاده شود (برای وضوح فقط ۲۰ مولفه اول بردار را نشان می‌دهیم):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

نکته‌ی جالب دربارهٔ جاسازی‌های معنایی این است که می‌توانید رمزنگاری برداری را دستکاری کنید تا معنای آن تغییر کند. برای مثال، می‌توانیم بخواهیم کلمه‌ای پیدا کنیم که نمایش برداری‌اش تا حد امکان به کلمات *king* و *woman* نزدیک باشد و از کلمهٔ *man* دور باشد:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

هر دو روش CBoW و Skip-Grams «تعبیه‌های پیش‌بینی‌کننده» هستند، به این معنا که تنها زمینه‌های محلی را در نظر می‌گیرند. Word2Vec از زمینه‌ی جهانی بهره نمی‌برد.

**FastText** بر اساس Word2Vec ساخته شده است و با یادگیری نمایش‌های برداری برای هر کلمه و n-گرام‌های کاراکتری موجود در هر کلمه، کار می‌کند. مقادیر نمایندگی‌ها سپس در هر مرحله آموزش به یک بردار متوسط تبدیل می‌شوند. در حالی که این امر محاسبات اضافی زیادی به پیش‌آموزش اضافه می‌کند، اما امکان کدگذاری اطلاعات زیرکلمه‌ای را در تعبیه‌های کلمه فراهم می‌سازد.

روش دیگری به نام **GloVe** از ایده ماتریس هم‌وقوعی استفاده می‌کند و با استفاده از روش‌های عصبی ماتریس هم‌وقوعی را به بردارهای کلمه‌ای غیرخطی و بیانگرتر تجزیه می‌کند.

می‌توانید با تغییر تعبیه‌ها به FastText و GloVe در این مثال بازی کنید، زیرا gensim از چندین مدل مختلف تعبیه کلمه پشتیبانی می‌کند.


## استفاده از جاسازی‌های از پیش آموزش‌دیده شده در PyTorch

ما می‌توانیم مثال بالا را اصلاح کنیم تا ماتریس در لایه جاسازی خود را با جاسازی‌های معنایی مانند Word2Vec پیش‌پر کنیم. باید در نظر داشته باشیم که واژگان جاسازی از پیش آموزش‌دیده شده و مجموعه متن ما احتمالاً مطابقت نخواهند داشت، بنابراین وزن‌های کلمات مفقود را با مقادیر تصادفی مقداردهی اولیه می‌کنیم:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


حالا بیایید مدل خود را آموزش دهیم. توجه داشته باشید که زمانی که طول می‌کشد تا مدل آموزش داده شود، به طور قابل توجهی بیشتر از مثال قبلی است، به دلیل اندازه بزرگ‌تر لایه جاسازی، و در نتیجه تعداد بسیار بیشتری پارامتر دارد. همچنین، به همین دلیل، ممکن است نیاز باشد مدل خود را روی نمونه‌های بیشتری آموزش دهیم اگر بخواهیم از بیش‌برازش جلوگیری کنیم.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

در مورد ما افزایش بزرگی در دقت مشاهده نمی‌کنیم، که احتمالاً به دلیل واژگان کاملاً متفاوت است. 
برای غلبه بر مشکل واژگان متفاوت، می‌توانیم از یکی از راه‌حل‌های زیر استفاده کنیم:
* مدل word2vec را روی واژگان خود دوباره آموزش دهیم
* داده‌های خود را با واژگان مدل پیش‌آموزش‌دیده word2vec بارگذاری کنیم. واژگانی که برای بارگذاری داده‌ها استفاده می‌شود می‌تواند هنگام بارگذاری مشخص شود.

رویکرد دوم به نظر آسان‌تر می‌آید، به ویژه چون چارچوب PyTorch `torchtext` پشتیبانی داخلی از embedding ها دارد. ما می‌توانیم، برای مثال، لغت‌نامه مبتنی بر GloVe را به شکل زیر نمونه‌سازی کنیم:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Loaded vocabulary has the following basic operations:
* `vocab.stoi` dictionary allows us to convert word into its dictionary index
* `vocab.itos` does the opposite - converts number into word
* `vocab.vectors` is the array of embedding vectors, so to get the embedding of a word `s` we need to use `vocab.vectors[vocab.stoi[s]]`

Here is the example of manipulating embeddings to demonstrate the equation **kind-man+woman = queen** (I had to tweak the coefficient a bit to make it work):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

برای آموزش دسته‌بند با استفاده از آن تعبیه‌ها، ابتدا باید داده‌های خود را با استفاده از واژگان GloVe رمزگذاری کنیم:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

همان‌طور که در بالا دیدیم، تمام جاسازی‌های برداری در ماتریس `vocab.vectors` ذخیره شده‌اند. این کار بارگذاری آن وزن‌ها به وزن‌های لایه جاسازی را با استفاده از کپی ساده فوق‌العاده آسان می‌کند:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

حال بیایید مدل خود را آموزش دهیم و ببینیم آیا نتایج بهتری می‌گیریم:  


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

یکی از دلایلی که تغییر قابل توجهی در دقت مشاهده نمی‌کنیم این است که برخی از کلمات در دیتاست ما در واژگان از پیش آموزش‌دیده‌شده GloVe وجود ندارند و بنابراین اساساً نادیده گرفته می‌شوند. برای غلبه بر این موضوع، می‌توانیم تعبیه‌های خود را روی دیتاست خود آموزش دهیم.


## جاسازی‌های متنی زمینه‌ای

یکی از محدودیت‌های کلیدی بازنمایی‌های جاسازی آموزش‌دیده سنتی مانند Word2Vec مسئله تشخیص معنی کلمه است. در حالی که جاسازی‌های آموزش‌دیده می‌توانند بخشی از معنی کلمات را در زمینه به دست آورند، هر معنی ممکن برای یک کلمه در همان جاسازی رمزگذاری می‌شود. این می‌تواند در مدل‌های پایین‌دستی مشکل ایجاد کند، زیرا بسیاری از کلمات مانند کلمه «play» بسته به زمینه‌ای که در آن استفاده می‌شوند معانی متفاوتی دارند.

برای مثال، کلمه «play» در آن دو جمله متفاوت معانی کاملاً متفاوتی دارد:
- من به یک **نمایش** در تئاتر رفتم.
- جان می‌خواهد با دوستانش **بازی** کند.

جاسازی‌های آموزش‌دیده بالا هر دو این معانی کلمه «play» را در یک جاسازی نشان می‌دهند. برای رفع این محدودیت، نیاز است جاسازی‌هایی بسازیم که بر اساس **مدل زبانی** ساخته شده‌اند، مدلی که روی یک مجموعه بزرگی از متن آموزش دیده و *می‌داند* چگونه کلمات می‌توانند در زمینه‌های مختلف کنار هم قرار بگیرند. بحث درباره جاسازی‌های زمینه‌ای خارج از محدوده این آموزش است، اما وقتی درباره مدل‌های زبانی در واحد بعدی صحبت کنیم دوباره به آن‌ها باز خواهیم گشت.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
